# Sakura-Galtransl-14B-v3.8-Q5_K_S Local Colab Batch Translation
Run this notebook in Google Colab to deploy the Sakura translation model and run the EPUB translator directly inside Colab. No internal tunneling is needed.

## 1. Setup Environment (llama.cpp & EPUB-Translator)

In [ ]:
!echo "Installing dependencies and building llama.cpp..."
!git clone https://github.com/ggerganov/llama.cpp.git
!cd llama.cpp && cmake -B build -DGGML_CUDA=ON && cmake --build build --config Release -j 1

!echo "Downloading Sakura GGUF model..."
import huggingface_hub
huggingface_hub.hf_hub_download(repo_id="SakuraLLM/Sakura-GalTransl-14B-v3.8", filename="Sakura-Galtransl-14B-v3-Q5_K_S.gguf", local_dir=".", local_dir_use_symlinks=False)

!echo "Cloning and installing EPUB-Bilingual-Translator..."
# 替换为您的 GitHub 仓库地址
!git clone https://github.com/yukari502/EPUB-Bilingual-Translator.git
!cd EPUB-Bilingual-Translator && pip install -e .

## 2. Start Model Server (Background)

In [ ]:
import subprocess
import time
import os

# 寻找编译好的 llama-server 路径
server_bin = "./llama.cpp/build/bin/llama-server"
if not os.path.exists(server_bin):
    server_bin = "./llama.cpp/build/llama-server"
if not os.path.exists(server_bin):
    raise FileNotFoundError("无法找到 llama-server 可执行文件，可能是编译失败了！")

# Start llama.cpp server in the background
print(f"Starting {server_bin}...")
server_cmd = f"{server_bin} -m Sakura-Galtransl-14B-v3-Q5_K_S.gguf -c 4096 --host 127.0.0.1 --port 8000 -ngl 999"
subprocess.Popen(server_cmd, shell=True, stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)

time.sleep(10)  # Wait for the model to load in VRAM
print("Server is running locally at http://127.0.0.1:8000")


## 3. Batch Translate EPUBs
Put your `.epub` files inside the `books/` folder in Colab before running this cell. The translated files will be saved in `translated/`.

In [ ]:
!mkdir -p books
!mkdir -p translated

!echo "Start batch translation..."
!epub-translator ./books/ ./translated/ --provider custom --model sakura-14b --api-url http://127.0.0.1:8000/v1/chat/completions

!echo "Translation finished. Check the 'translated' folder!"